### Exploratory Data Analysis

In [1]:
import os
import json
import re

import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

# Get lesion volumes

In [5]:
def get_information(raw_data_path:str, dataset:str) ->pd.DataFrame:
    if dataset == "isles22":
        dirs = sorted(os.listdir(raw_data_path))
        dirs = dirs[5:]
    else:
        dirs = sorted(os.listdir(raw_data_path + "/raw_data"))

    information = []
    id = 0
    for folder in dirs:
        if dataset == "isles22":
            path = f"{raw_data_path}/{folder}/ses-0001/dwi/{folder}_ses-0001_dwi.nii.gz"
            gt_path  = f"{raw_data_path}/derivatives/{folder}/ses-0001/{folder}_ses-0001_msk.nii.gz"
        else:
            path = f"{raw_data_path}/raw_data/{folder}/ses-02/{folder}_ses-02_dwi.nii.gz"
            gt_path = f"{raw_data_path}/derivatives/{folder}/ses-02/{folder}_ses-02_lesion-msk.nii.gz"
        dwi_image = nib.load(path)
        msk_image = nib.load(gt_path).get_fdata()
        img_shape = dwi_image.shape
        voxel_size = dwi_image.header.get_zooms()
        voxel_volume = np.prod(voxel_size)/1000
        lesion_volume = np.sum(msk_image) * voxel_volume

        id = re.search(r'(\d+)$', folder)
        information.append({"Id":int(id.group(1)), "lesion_vol": lesion_volume, "shape":img_shape, "voxel size":voxel_size})
        # id+=1
        
    information = pd.DataFrame(information)
    return information
    

In [10]:
isles_22 = "/home/dayan/Documents/Datasets/stroke_segmentation/ISLES-2022"
voxel_sizes_isles22 = get_information(isles_22, "isles22")
# voxel_sizes_isles22.tail()

print("\n shape counts\n", voxel_sizes_isles22["shape"].value_counts())
print("\n voxel counts\n", voxel_sizes_isles22["voxel size"].value_counts())


 shape counts
 shape
(112, 112, 73)    93
(112, 112, 72)    92
(128, 128, 25)    27
(192, 192, 30)    23
(128, 128, 72)     2
(256, 256, 73)     2
(115, 115, 75)     1
(115, 115, 28)     1
(130, 130, 65)     1
(170, 170, 33)     1
(112, 112, 74)     1
(112, 112, 75)     1
(115, 115, 70)     1
(112, 112, 76)     1
(128, 128, 27)     1
(122, 122, 63)     1
(128, 128, 30)     1
Name: count, dtype: int64

 voxel counts
 voxel size
(2.0, 2.0, 2.0)                193
(1.796875, 1.796875, 4.8)       29
(1.1458334, 1.1458334, 4.8)     23
(0.875, 0.875, 2.0)              2
(2.0, 2.0, 5.0)                  1
(1.3529412, 1.3529412, 4.4)      1
(1.9672132, 1.9672132, 2.0)      1
Name: count, dtype: int64


In [14]:
print("Difficult samples\n", voxel_sizes_isles22[voxel_sizes_isles22["Id"] == 233])
print("Difficult samples\n", voxel_sizes_isles22[voxel_sizes_isles22["Id"] == 234])

Difficult samples
       Id  lesion_vol           shape       voxel size
232  233       0.064  (112, 112, 72)  (2.0, 2.0, 2.0)
Difficult samples
       Id  lesion_vol           shape       voxel size
233  234       0.232  (112, 112, 72)  (2.0, 2.0, 2.0)


In [20]:
# smallest sample by volume
print("Smallest sample by volume", voxel_sizes_isles22[voxel_sizes_isles22["lesion_vol"] < 0.1])

Smallest sample by volume       Id  lesion_vol           shape       voxel size
5      6       0.040  (112, 112, 72)  (2.0, 2.0, 2.0)
116  117       0.048  (112, 112, 73)  (2.0, 2.0, 2.0)
143  144       0.080  (112, 112, 73)  (2.0, 2.0, 2.0)
149  150       0.000  (112, 112, 75)  (2.0, 2.0, 2.0)
150  151       0.000  (112, 112, 73)  (2.0, 2.0, 2.0)
169  170       0.000  (112, 112, 72)  (2.0, 2.0, 2.0)
232  233       0.064  (112, 112, 72)  (2.0, 2.0, 2.0)


In [21]:
isles_24_train = "/home/dayan/Documents/Datasets/stroke_segmentation/isles24_train_b1/isles24_train_a"
isles_24_test = "/home/dayan/Documents/Datasets/stroke_segmentation/isles24_train_b2/isles24_train_b"

voxel_sizes_train = get_information(isles_24_train, "isles24")
voxel_sizes_test = get_information(isles_24_test, "isles24")
voxel_sizes_test.head()

,Id,lesion_vol,shape,voxel size
0,3,0.745924,"(512, 633, 70)","(0.41015625, 0.41015625, 2.0)"
1,5,56.033094,"(512, 536, 69)","(0.41015625, 0.41015625, 2.0)"
2,6,0.000000,"(512, 544, 64)","(0.41015625, 0.41015625, 2.0)"
3,14,222.129432,"(512, 586, 74)","(0.3828125, 0.3828125, 2.0)"
4,17,138.634274,"(512, 602, 65)","(0.46875, 0.46875, 2.0)"


I define the lesion categories as follows:
 - small: lesion volume < 5ml
 - medium: lesion volume >=5ml and < 20ml
 - large: lesion volume >= 20ml

In [7]:
def vol_count(arr):
    small_lesions = arr[(arr["lesion_vol"] < 5.0)]
    medium_lesions = arr[(arr["lesion_vol"] >= 5.0) & (arr["lesion_vol"] < 20.0)]
    large_lesions = arr[arr["lesion_vol"] >= 20]

    return small_lesions, medium_lesions, large_lesions

#define splits for isles'22
train = voxel_sizes_isles22.iloc[:174, :]
validation = voxel_sizes_isles22.iloc[175:199, :]
test = voxel_sizes_isles22.iloc[200:, :]

small22tr, medium22tr, large22tr = vol_count(train)
small22vl, medium22vl, large22vl = vol_count(validation)
small22tst, medium22tst, large22tst = vol_count(test)


print("Lesion size breakdown Isles22")
print(f"small lesions: {small22tr.shape[0] + small22vl.shape[0]+small22tst.shape[0]}")
print(f"medium lesions: {medium22tr.shape[0] + medium22vl.shape[0]+medium22tst.shape[0]}")
print(f"large lesions: {large22tr.shape[0] + large22vl.shape[0] + large22tst.shape[0]}")


small24tst, medium24tst, large24tst = vol_count(voxel_sizes_test)
small24trn, medium24trn, large24trn = vol_count(voxel_sizes_train)
print("Lesion size breakdown Isles24")
print(f"small lesions: {small24trn.shape[0] + small24tst.shape[0]}")
print(f"medium lesions: {medium24trn.shape[0] + medium24tst.shape[0]}")
print(f"large lesions: {large24trn.shape[0] + large24tst.shape[0]}")

Lesion size breakdown Isles22
small lesions: 103
medium lesions: 80
large lesions: 65
Lesion size breakdown Isles24
small lesions: 46
medium lesions: 50
large lesions: 53


Check current train-test split volume distribution on Isles22.

In [8]:
print("Train")
print(f"small lesions: {small22tr.shape[0]}")
print(f"medium lesions: {medium22tr.shape[0]}")
print(f"large lesions: {large22tr.shape[0]}")



print("\nValidation")
print(f"small lesions: {small22vl.shape[0]}")
print(f"medium lesions: {medium22vl.shape[0]}")
print(f"large lesions: {large22vl.shape[0]}")

print("\nTest")
print(f"small lesions: {small22tst.shape[0]}")
print(f"medium lesions: {medium22tst.shape[0]}")
print(f"large lesions: {large22tst.shape[0]}")

Train
small lesions: 91
medium lesions: 39
large lesions: 44

Validation
small lesions: 3
medium lesions: 18
large lesions: 3

Test
small lesions: 9
medium lesions: 23
large lesions: 18


Train-Test Volume Split On Isles'24

In [9]:
print("Train")
print(f"small lesions: {small24trn.shape[0]}")
print(f"medium lesions: {medium24trn.shape[0]}")
print(f"large lesions: {large24trn.shape[0]}")

print("Test")
print(f"small lesions: {small24tst.shape[0]}")
print(f"medium lesions: {medium24tst.shape[0]}")
print(f"large lesions: {large24tst.shape[0]}")

Train
small lesions: 37
medium lesions: 38
large lesions: 40
Test
small lesions: 9
medium lesions: 12
large lesions: 13


### Results Analysis

In [179]:
def load_jsonl(file_path:str):

    data = []
    with open(file_path) as f:
        for line in f:
            data.append(json.loads(line))
        
    df = pd.DataFrame(data)
    return df

#isles'22
unet = "../../metrics/test/14_isles22_individual_metrics.jsonl"
unetr = "../../metrics/test/15_isles22_individual_metrics.jsonl"
segres = "../../metrics/test/16_isles22_individual_metrics.jsonl"
higher557 = "../../metrics/test/debug_individual.jsonl"
swin = "../../metrics/test/swin_22_individual.jsonl"
vit = "../../metrics/test/vit_22_individual.jsonl"
attnunet = "../../metrics/test/attnunet_22_individual.jsonl"
unetpp = "../../metrics/test/unetpp_22_individual.jsonl"

unet = load_jsonl(unet)
unetr = load_jsonl(unetr)
segres = load_jsonl(segres)
my_model = load_jsonl(higher557)
swin = load_jsonl(swin)
vit = load_jsonl(vit)
attnunet = load_jsonl(attnunet)
unetpp = load_jsonl(unetpp)

#ablation isles'22
# convblock_ablation = "../../metrics/test/5_individual_ablation_convblock.jsonl"
bottleneck_conv_layer = "../../metrics/test/my_model_bottleneck_convol_22_individual.jsonl"
bottleneck_without = "../../metrics/test/my_model_bottleneck_nobottleneck_22_individual.jsonl"
bottlenck_experimentation = "../../metrics/test/debug2_individual.jsonl"

# convab = load_jsonl(convblock_ablation)
bottle_conv = load_jsonl(bottleneck_conv_layer)
wobottleab =load_jsonl(bottleneck_without)
bottleneck_exp = load_jsonl(bottlenck_experimentation)


#isles'24
my_model24 = "../../metrics/test/0_individual_higher557_24.jsonl"
unet24 = "../../metrics/test/1_isles22_individual_metrics.jsonl"
unetr24 = "../../metrics/test/2_isles22_individual_metrics.jsonl"
segres24 = "../../metrics/test/3_isles22_individual_metrics.jsonl"
swin_24 = "../../metrics/test/swin_24_individual.jsonl"
vit_24 = "../../metrics/test/vit_24_individual.jsonl"
attnunet_24 = "../../metrics/test/attnunet_24_individual.jsonl"
unetpp_24 = "../../metrics/test/unetpp_24_individual.jsonl"

my_model24 = load_jsonl(my_model24)
unet24 = load_jsonl(unet24)
unetr24 = load_jsonl(unetr24)
segres24 = load_jsonl(segres24)
swin24 = load_jsonl(swin_24)
vit24 = load_jsonl(vit_24)
attnunet24 = load_jsonl(attnunet_24)
unetpp24 = load_jsonl(unetpp_24)


### Train 22, Test 24
my_model_tr22_tst24 = "../../metrics/test/0_individual_higher557_tr22_tst24.jsonl"
unet_tr22_tst24 = "../../metrics/test/14_individual_higher557_tr22_tst24.jsonl"
unetr_tr22_tst24 = "../../metrics/test/15_individual_higher557_tr22_tst24.jsonl"
segres_tr22_tst24 = "../../metrics/test/16_individual_higher557_tr22_tst24.jsonl"

my_model_tr22_tst24 = load_jsonl(my_model_tr22_tst24)
unet_tr22_tst24 = load_jsonl(unet_tr22_tst24)
unetr_tr22_tst24 = load_jsonl(unetr_tr22_tst24)
segres_tr22_tst24 = load_jsonl(segres_tr22_tst24)


### Train 24, Test 22
my_model_tr24_tst22 = "../../metrics/test/0_individual_higher557_tr24_tst22.jsonl"
unet_tr24_tst22 = "../../metrics/test/1_individual_tr24_tst22.jsonl"
unetr_tr24_tst22 = "../../metrics/test/2_individual_tr24_tst22.jsonl"
segres_tr24_tst22 = "../../metrics/test/3_individual_higher557_tr24_tst22.jsonl"

my_model_tr24_tst22 = load_jsonl(my_model_tr24_tst22)
unet_tr24_tst22 = load_jsonl(unet_tr24_tst22)
unetr_tr24_tst22 = load_jsonl(unetr_tr24_tst22)
segres_tr24_tst22 = load_jsonl(segres_tr24_tst22)



# dilation analysis ablation
wot11 = load_jsonl("../../metrics/test/0_individual_higher557_22_dilation_ablation11.jsonl")
wot12 = load_jsonl("../../metrics/test/0_individual_higher557_22_dilation_ablation12.jsonl")
wot13 = load_jsonl("../../metrics/test/0_individual_higher557_22_dilation_ablation13.jsonl")
wot21 = load_jsonl("../../metrics/test/0_individual_higher557_22_dilation_ablation21.jsonl")
wot22 = load_jsonl("../../metrics/test/0_individual_higher557_22_dilation_ablation22.jsonl")
wot23 = load_jsonl("../../metrics/test/0_individual_higher557_22_dilation_ablation23.jsonl")


In [180]:
test_id = [i for i in range(200, 251)]
train_id = [i for i in range(1, 175) if i != 150 and i != 151 and i != 170]
val_id = [i for i in range(175, 200)]

unet["Id"] = test_id
unetr["Id"] = test_id
segres["Id"] = test_id
swin["Id"] = test_id
attnunet["Id"] = test_id
vit["Id"] = test_id
bottle_conv["Id"] = test_id
wobottleab["Id"] = test_id
bottleneck_exp["Id"] = test_id
unetpp["Id"] = test_id
my_model["Id"] = test_id




In [181]:
my_model_tr22_tst24["Id"] = voxel_sizes_test["Id"]
unet_tr22_tst24["Id"] = voxel_sizes_test["Id"]
unetr_tr22_tst24["Id"] = voxel_sizes_test["Id"]
segres_tr22_tst24["Id"] = voxel_sizes_test["Id"]

In [182]:
my_model_tr24_tst22["Id"] = test_id
unet_tr24_tst22["Id"] = test_id
unetr_tr24_tst22["Id"] = test_id
segres_tr24_tst22["Id"] = test_id

In [183]:
wot11["Id"] = test_id
wot12["Id"] = test_id
wot13["Id"] = test_id
wot21["Id"] = test_id
wot22["Id"] = test_id
wot23["Id"] = test_id

In [184]:
my_model24["Id"] = voxel_sizes_test["Id"]
unet24["Id"] = voxel_sizes_test["Id"]
unetr24["Id"] = voxel_sizes_test["Id"]
segres24["Id"] = voxel_sizes_test["Id"]
swin24["Id"] = voxel_sizes_test["Id"]
attnunet24["Id"] = voxel_sizes_test["Id"]
vit24["Id"] = voxel_sizes_test["Id"]
unetpp24["Id"] = voxel_sizes_test["Id"]

unet24.head()

,dice,jc,hd,asd,assd,precision,se,sp,F1,Id
0,0.166743,0.090954,155.000000,95.564937,85.923802,0.093622,0.761442,0.999302,0.105263,3
1,0.791217,0.654557,5.385165,1.735467,1.733756,0.830431,0.755540,0.998626,0.352423,5
2,0.839195,0.722943,7.280110,3.657669,2.526962,0.951796,0.750419,0.998654,0.328767,6
3,0.428158,0.272393,110.896573,40.517079,18.056790,0.785547,0.294276,0.998713,0.000000,14
4,0.295505,0.173368,89.094332,41.506878,32.838840,0.187468,0.697431,0.999504,0.142857,17


Verify Ids

In [185]:
with open('test_22.list', 'r') as file:
    test = [line.strip() for line in file]

with open('train_22.list', 'r') as file:
    train = [line.strip() for line in file]

print(f'There are {len(train)} file in the training set and {len(test)} in the testing set.')

There are 196 file in the training set and 51 in the testing set.


### Getting lesion volumes

In [186]:
no_lesion = voxel_sizes_isles22[voxel_sizes_isles22["lesion_vol"] == 0]
print(no_lesion.shape)
print(f"Id of no lesion samples:\n{no_lesion.Id.to_string(index=False)}")

have_lesion = voxel_sizes_isles22[voxel_sizes_isles22["lesion_vol"] > 0]
print(have_lesion.shape)

(3, 2)
Id of no lesion samples:
150
151
170
(247, 2)


In [187]:
no_lesion_24 = voxel_sizes_test[voxel_sizes_test["lesion_vol"] == 0]
print(no_lesion_24.shape)
print(f"Id of no lesion samples:\n{no_lesion_24.Id.to_string(index=False)}")

have_lesion_24 = voxel_sizes_test[voxel_sizes_test["lesion_vol"] > 0]
print(have_lesion_24.shape)

(1, 2)
Id of no lesion samples:
6
(33, 2)


In [188]:
have_lesion_val = have_lesion[171:196]
have_lesion_test = have_lesion[196:]
have_lesion_test = have_lesion_test.astype(int)

unet = pd.merge(left=unet, right=have_lesion_test, on="Id")
unetr = pd.merge(left=unetr, right=have_lesion_test, on="Id")
segres = pd.merge(left=segres, right=have_lesion_test, on="Id")
my_model = pd.merge(left=my_model, right=have_lesion_test, on="Id")
swin = pd.merge(left=swin, right=have_lesion_test, on="Id")
vit = pd.merge(left=vit, right=have_lesion_test, on="Id")
attnunet = pd.merge(left=attnunet, right=have_lesion_test, on="Id")
unetpp = pd.merge(left=unetpp, right=have_lesion_test, on="Id")

attnunet.head()

,dice,hd,precision,F1,Id,lesion_vol
0,0.150823,32.972702,0.690411,0.000000,200,8
1,0.624285,7.000000,0.900419,0.474576,201,147
2,0.785705,2.828427,0.804807,0.666667,202,7
3,0.692861,2.236068,0.723724,1.000000,203,1
4,0.329312,12.000000,0.894602,0.000000,204,13


In [189]:
bottle_conv = pd.merge(left=bottle_conv, right=have_lesion_test, on="Id")
wobottleab = pd.merge(left=wobottleab, right=have_lesion_test, on="Id")
bottleneck_exp = pd.merge(left=bottleneck_exp, right=have_lesion_test, on="Id")

In [190]:
my_model_tr22_tst24 = pd.merge(left=my_model_tr22_tst24, right=have_lesion_24, on="Id")
unet_tr22_tst24 = pd.merge(left=unet_tr22_tst24, right=have_lesion_24, on="Id")
unetr_tr22_tst24 = pd.merge(left=unetr_tr22_tst24, right=have_lesion_24, on="Id")
segres_tr22_tst24 = pd.merge(left=segres_tr22_tst24, right=have_lesion_24, on="Id")

In [174]:
my_model_tr24_tst22 = pd.merge(left=my_model_tr24_tst22, right=have_lesion_test, on="Id")
unet_tr24_tst22 = pd.merge(left=unet_tr24_tst22, right=have_lesion_test, on="Id")
unetr_tr24_tst22 = pd.merge(left=unetr_tr24_tst22, right=have_lesion_test, on="Id")
segres_tr24_tst22 = pd.merge(left=segres_tr24_tst22, right=have_lesion_test, on="Id")

In [176]:
wot11 = pd.merge(left=wot11, right=have_lesion_test, on="Id")
wot12 = pd.merge(left=wot12, right=have_lesion_test, on="Id")
wot13 = pd.merge(left=wot13, right=have_lesion_test, on="Id")
wot21 = pd.merge(left=wot21, right=have_lesion_test, on="Id")
wot22 = pd.merge(left=wot22, right=have_lesion_test, on="Id")
wot23 = pd.merge(left=wot23, right=have_lesion_test, on="Id")

In [191]:
my_model24 = pd.merge(left=my_model24, right=have_lesion_24, on="Id")
unet24 = pd.merge(left=unet24, right=have_lesion_24, on="Id")
unetr24 = pd.merge(left=unetr24, right=have_lesion_24, on="Id")
segres24 = pd.merge(left=segres24, right=have_lesion_24, on="Id")
swin24 = pd.merge(left=swin24, right=have_lesion_24, on="Id")
attnunet24 = pd.merge(left=attnunet24, right=have_lesion_24, on="Id")
vit24 = pd.merge(left=vit24, right=have_lesion_24, on="Id")
unetpp24 = pd.merge(left=unetpp24, right=have_lesion_24, on="Id")

segres24.head()

,dice,jc,hd,asd,assd,precision,se,sp,F1,Id,lesion_vol
0,0.561785,0.390613,39.804521,6.468601,5.091966,0.478092,0.680999,0.999930,0.444444,3,0.745924
1,0.762731,0.616463,7.348469,0.615071,2.186089,0.905072,0.659077,0.999384,0.426036,5,56.033094
2,0.498098,0.331644,112.004464,41.576670,18.596750,0.779425,0.365995,0.998340,0.529412,14,222.129432
3,0.626247,0.455866,3.000000,2.723153,1.833234,0.524759,0.776403,0.999885,0.750000,17,138.634274
4,0.513157,0.345132,13.964240,2.159120,3.310000,0.699855,0.405092,0.999859,0.600924,20,1.119385


### Test of Isles'22 F1 score calculation 

In [177]:
thresholds = [
    ('very small', 0, 5),
    ('medium',     5, 15),
    ('large',      15, 9999)
]

# models = [
#     ('wot11', wot11),
#     ('wot12', wot12),
#     ('wot13', wot13),
#     ('wot21', wot21),
#     ('wot22', wot22),
#     ('wot23', wot23),
#     ('higher557', higher557)]

models = [
    ('my_model', my_model),
    ('unet', unet),
    ('unetr', unetr),
    ('segres', segres),
    # ('conv ablation', convab),
    # ('bottleneck ablation', bottleab),
    ('swin', swin),
    ('vit', vit),
    ('attnunet', attnunet),
    ('unetpp', unetpp)
]

for label, low, high in thresholds:
    for name, df in models :
        sub = df[(df['lesion_vol'] >= low) & 
                 (df['lesion_vol'] <  high)]
        if len(sub) > 0:
            print(f"{label} ({low}-{high}ml) {name} "
                  f"n={len(sub):2d}  "
                  f"Dice={sub['dice'].mean():.3f}  "
                  f"HD95={sub["hd"].mean():.3f}   "
                  f"F1={sub['F1'].mean():.3f}")
    print()

very small (0-5ml) my_model n= 9  Dice=0.545  HD95=23.101   F1=0.621
very small (0-5ml) unet n= 9  Dice=0.465  HD95=45.759   F1=0.410
very small (0-5ml) unetr n= 9  Dice=0.453  HD95=57.001   F1=0.402
very small (0-5ml) segres n= 9  Dice=0.510  HD95=55.981   F1=0.457
very small (0-5ml) swin n= 9  Dice=0.543  HD95=16.925   F1=0.633
very small (0-5ml) vit n= 9  Dice=0.109  HD95=73.502   F1=0.078
very small (0-5ml) attnunet n= 9  Dice=0.458  HD95=41.253   F1=0.560
very small (0-5ml) unetpp n= 9  Dice=0.523  HD95=34.123   F1=0.654

medium (5-15ml) my_model n=19  Dice=0.762  HD95=5.931   F1=0.688
medium (5-15ml) unet n=19  Dice=0.707  HD95=17.460   F1=0.472
medium (5-15ml) unetr n=19  Dice=0.666  HD95=14.160   F1=0.453
medium (5-15ml) segres n=19  Dice=0.733  HD95=24.202   F1=0.457
medium (5-15ml) swin n=19  Dice=0.728  HD95=6.021   F1=0.678
medium (5-15ml) vit n=19  Dice=0.454  HD95=38.836   F1=0.288
medium (5-15ml) attnunet n=19  Dice=0.621  HD95=14.950   F1=0.476
medium (5-15ml) unetpp n=

## Bottleneck ablation 

In [192]:
models = [
    ('Original', my_model),
    ('Convolution layer inplace of bottleneck', bottle_conv),
    ('Without Bottleneck', wobottleab),
    ('bottleneck experimentation', bottleneck_exp)
]

for label, low, high in thresholds:
    for name, df in models :
        sub = df[(df['lesion_vol'] >= low) & 
                 (df['lesion_vol'] <  high)]
        if len(sub) > 0:
            print(f"{label} ({low}-{high}ml) {name} "
                  f"n={len(sub):2d}  "
                  f"Dice={sub['dice'].mean():.3f}  "
                  f"HD95={sub["hd"].mean():.3f}   "
                  f"F1={sub['F1'].mean():.3f}")
    print()

very small (0-5ml) Original n= 9  Dice=0.545  HD95=23.101   F1=0.621
very small (0-5ml) Convolution layer inplace of bottleneck n= 9  Dice=0.530  HD95=24.211   F1=0.609
very small (0-5ml) Without Bottleneck n= 9  Dice=0.554  HD95=28.908   F1=0.551
very small (0-5ml) bottleneck experimentation n= 9  Dice=0.550  HD95=19.728   F1=0.583

medium (5-15ml) Original n=19  Dice=0.762  HD95=5.931   F1=0.688
medium (5-15ml) Convolution layer inplace of bottleneck n=19  Dice=0.770  HD95=5.945   F1=0.633
medium (5-15ml) Without Bottleneck n=19  Dice=0.773  HD95=6.006   F1=0.634
medium (5-15ml) bottleneck experimentation n=19  Dice=0.755  HD95=9.486   F1=0.638

large (15-9999ml) Original n=23  Dice=0.813  HD95=6.351   F1=0.582
large (15-9999ml) Convolution layer inplace of bottleneck n=23  Dice=0.811  HD95=6.649   F1=0.573
large (15-9999ml) Without Bottleneck n=23  Dice=0.813  HD95=6.744   F1=0.553
large (15-9999ml) bottleneck experimentation n=23  Dice=0.813  HD95=6.352   F1=0.597



In [75]:
models = [('my_model', my_model_tr22_tst24),
                     ('unet', unet_tr22_tst24),
                     ('unetr', unetr_tr22_tst24),
                     ('segres', segres_tr22_tst24)]

for label, low, high in thresholds:
    for name, df in models :
        sub = df[(df['lesion_vol'] >= low) & 
                 (df['lesion_vol'] <  high)]
        if len(sub) > 0:
            print(f"{label} ({low}-{high}ml) {name} "
                  f"n={len(sub):2d}  "
                  f"Dice={sub['dice'].mean():.3f}  "
                  f"HD95={sub["hd"].mean():.3f}   "
                  f"Recall={sub["se"].mean():.3f}   "
                  f"F1={sub['F1'].mean():.3f}")
    print()

very small (0-5ml) my_model n= 8  Dice=0.604  HD95=42.683   Recall=0.612   F1=0.473
very small (0-5ml) unet n= 8  Dice=0.510  HD95=61.443   Recall=0.567   F1=0.321
very small (0-5ml) unetr n= 8  Dice=0.548  HD95=43.096   Recall=0.523   F1=0.346
very small (0-5ml) segres n= 8  Dice=0.510  HD95=60.627   Recall=0.711   F1=0.415

medium (5-15ml) my_model n=11  Dice=0.705  HD95=28.025   Recall=0.704   F1=0.477
medium (5-15ml) unet n=11  Dice=0.579  HD95=58.745   Recall=0.644   F1=0.286
medium (5-15ml) unetr n=11  Dice=0.535  HD95=33.209   Recall=0.502   F1=0.329
medium (5-15ml) segres n=11  Dice=0.622  HD95=67.640   Recall=0.855   F1=0.381

large (15-9999ml) my_model n=12  Dice=0.691  HD95=23.286   Recall=0.681   F1=0.569
large (15-9999ml) unet n=12  Dice=0.549  HD95=83.584   Recall=0.652   F1=0.247
large (15-9999ml) unetr n=12  Dice=0.598  HD95=34.426   Recall=0.570   F1=0.416
large (15-9999ml) segres n=12  Dice=0.622  HD95=43.478   Recall=0.784   F1=0.429



In [76]:
models = [('my_model', my_model_tr24_tst22),
                     ('unet', unet_tr24_tst22),
                     ('unetr', unetr_tr24_tst22),
                     ('segres', segres_tr24_tst22)]

for label, low, high in thresholds:
    for name, df in models :
        sub = df[(df['lesion_vol'] >= low) & 
                 (df['lesion_vol'] <  high)]
        if len(sub) > 0:
            print(f"{label} ({low}-{high}ml) {name} "
                  f"n={len(sub):2d}  "
                  f"Dice={sub['dice'].mean():.3f}  "
                  f"HD95={sub["hd"].mean():.3f}   "
                  f"Recall={sub["se"].mean():.3f}   "
                  f"F1={sub['F1'].mean():.3f}")
    print()

very small (0-5ml) my_model n= 9  Dice=0.471  HD95=52.009   Recall=0.482   F1=0.354
very small (0-5ml) unet n= 9  Dice=0.259  HD95=90.814   Recall=0.595   F1=0.125
very small (0-5ml) unetr n= 9  Dice=0.248  HD95=74.547   Recall=0.522   F1=0.170
very small (0-5ml) segres n= 9  Dice=0.373  HD95=71.259   Recall=0.450   F1=0.317

medium (5-15ml) my_model n=19  Dice=0.716  HD95=14.879   Recall=0.662   F1=0.508
medium (5-15ml) unet n=19  Dice=0.630  HD95=70.715   Recall=0.715   F1=0.188
medium (5-15ml) unetr n=19  Dice=0.538  HD95=61.440   Recall=0.670   F1=0.193
medium (5-15ml) segres n=19  Dice=0.686  HD95=23.442   Recall=0.654   F1=0.404

large (15-9999ml) my_model n=23  Dice=0.780  HD95=7.194   Recall=0.704   F1=0.495
large (15-9999ml) unet n=23  Dice=0.751  HD95=35.808   Recall=0.771   F1=0.223
large (15-9999ml) unetr n=23  Dice=0.720  HD95=23.064   Recall=0.722   F1=0.329
large (15-9999ml) segres n=23  Dice=0.786  HD95=14.386   Recall=0.732   F1=0.472



In [75]:
thresholds = [
    ('very small', 0, 5),
    ('medium',     5, 15),
    ('large',      15, 9999)
]

for label, low, high in thresholds:
    for name, df in [('my_model', my_model24),
                     ('unet', unet24),
                     ('unetr', unetr24),
                     ('segres', segres24),
                     ('swin24', swin24),
                     ('attnunet24', attnunet24),
                     ('vit24', vit24),
                     ('unetpp24', unetpp24)]:
        sub = df[(df['lesion_vol'] >= low) & 
                 (df['lesion_vol'] <  high)]
        if len(sub) > 0:
            print(f"{label} ({low}-{high}ml) {name} "
                  f"n={len(sub):2d}  "
                  f"Dice={sub['dice'].mean():.3f}  "
                  f"HD95={sub["hd"].mean():.3f}   "
                  f"F1={sub['F1'].mean():.3f}")
    print()

very small (0-5ml) my_model n= 8  Dice=0.539  HD95=64.328   F1=0.415
very small (0-5ml) unet n= 8  Dice=0.443  HD95=104.156   F1=0.210
very small (0-5ml) unetr n= 8  Dice=0.501  HD95=63.817   F1=0.262
very small (0-5ml) segres n= 8  Dice=0.587  HD95=60.055   F1=0.468
very small (0-5ml) swin24 n= 8  Dice=0.554  HD95=37.138   F1=0.444
very small (0-5ml) attnunet24 n= 8  Dice=0.346  HD95=84.535   F1=0.143
very small (0-5ml) vit24 n= 8  Dice=0.233  HD95=72.605   F1=0.089
very small (0-5ml) unetpp24 n= 8  Dice=0.569  HD95=63.300   F1=0.515

medium (5-15ml) my_model n=11  Dice=0.681  HD95=58.081   F1=0.510
medium (5-15ml) unet n=11  Dice=0.557  HD95=88.479   F1=0.253
medium (5-15ml) unetr n=11  Dice=0.660  HD95=52.288   F1=0.328
medium (5-15ml) segres n=11  Dice=0.656  HD95=65.584   F1=0.473
medium (5-15ml) swin24 n=11  Dice=0.668  HD95=9.567   F1=0.524
medium (5-15ml) attnunet24 n=11  Dice=0.409  HD95=81.245   F1=0.135
medium (5-15ml) vit24 n=11  Dice=0.414  HD95=70.198   F1=0.139
medium (5